In [3]:
import pandas as pd
import numpy as np
import glob
import os
import gc
import hashlib
import pyarrow
from functions import *

Try

In [14]:
df = pd.read_csv('data_origin/omi_ntn/residential/VCN_1366809_1_2014_VALORI-RES.csv', sep = ';')

In [15]:
df.head()

,AREA,Regione,prov,2014_CodFitt,NTN 2014 fino a 50 mq,NTN 2014 50 -| 85 mq,NTN 2014 85 -| 115 mq,NTN 2014 115 -| 145 mq,NTN 2014 oltre 145,NTN 2014
0,Nord Ovest,Liguria,GE,A388,"21,83","50,67","39,49",13,"6,05","131,04"
1,Nord Ovest,Liguria,GE,A506,0,"3,5",3,"2,5",1,10
2,Nord Ovest,Liguria,GE,A658,2,"6,33","6,25","3,75",3,"21,33"
3,Nord Ovest,Liguria,GE,A922,"3,08","10,17","9,5",10,3,"35,75"
4,Nord Ovest,Liguria,GE,B067,0,2,"4,25",1,4,"11,25"


Load OMI data - read - concatenate

In [8]:
# Grab all CSV files
folder = "data_origin/omi_ntn/residential"

all_files = glob.glob(os.path.join(folder, "*.csv"))

out_dir = "datasets/omi_ntn"
os.makedirs(out_dir, exist_ok=True)

In [9]:
all_files

['data_origin/omi_ntn/residential\\VCN_1366809_1_2014_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366856_1_2015_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366857_1_2016_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366858_1_2017_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366859_1_2018_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366860_1_2019_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366861_1_2020_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366862_1_2021_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366864_1_2022_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366865_1_2023_VALORI-RES.csv',
 'data_origin/omi_ntn/residential\\VCN_1366866_1_2024_VALORI-RES.csv']

In [11]:
# Load ISTAT codes
df_istat = pd.read_csv('datasets/mun_istat_codes.csv')

# Concatenate all datasets into a single DataFrame
dfs = []

for i,f in enumerate(all_files):
    """
    Read every dataset in the folder as input.

    Extract year from the file name.

    Select relevant columns.

    Return datasets with updated istat codes. 
    """
    # Extract filename without extension
    filename = os.path.splitext(os.path.basename(f))[0]
    
    # Extract year from file name
    parts = filename.split("_")
    year = parts[-4]  # second to last part
    
    # Read CSV
    df = pd.read_csv(f, sep=';')
    
    # Strip whitespace and remove BOM from column names
    df.columns = df.columns.str.strip().str.replace('\ufeff','')

    # Year column
    df['year'] = year

    print(df.columns.tolist())
    print(year)

    # Translate columns names
    column_renames = {
        f'{year}_CodCom' : 'land_code', 
        f'NTN {year} fino a 50 mq' : 'less_50', 
        f'NTN {year} 50 -| 85 mq' : '50_85',
        f'NTN {year} 85 -| 115 mq' : '85_115',
        f'NTN {year} 115 -| 145 mq' : '115_145',
        f'NTN {year} oltre 145 mq' : 'more_145',
        f'NTN_{year}' : 'ntn'
    }

    df = df.rename(columns=column_renames)

    # Keep relevant columns
    df = df[['year', 'land_code', 'less_50', '50_85', '85_115', '115_145', 'more_145', 'ntn']]

    # Convert numeric columns to numeric type
    numeric_cols = ['less_50', '50_85', '85_115', '115_145', 'more_145', 'ntn']
    for col in numeric_cols:
        df[col] = df[col].astype('Int64')

    # Import ISTAT codes
    df = pd.merge(df, df_istat['land_code', 'mun_istat', 'mun_name', 'prov_name', 'reg_name'], on = 'land_code', how = 'left')

    add_zeroes(df, ['mun_istat'], 6)

    dfs.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(dfs, ignore_index=True)

['AREA', 'Regione', 'prov', '2014_CodFitt', 'NTN 2014 fino a 50 mq', 'NTN 2014 50 -| 85 mq', 'NTN 2014 85 -| 115 mq', 'NTN 2014 115 -| 145 mq', 'NTN 2014 oltre 145', 'NTN 2014', 'year']
1366809


KeyError: "['land_code', 'less_50', '50_85', '85_115', '115_145', 'more_145', 'ntn'] not in index"

Duplicated Istat codes

In [ ]:
# Count the number of duplicate listings
duplicates = final_df.value_counts(subset=[
    'mun_istat', 'zone', 'year_semester', 'condition', 'type'
    ])

duplicates = duplicates[duplicates > 1]

print("Number of duplicate listings for the same semester:", duplicates.sum())

Number of duplicate listings for the same semester: 544


In [ ]:
# Delete duplicate listings for the same semester - keep the first occurrence
final_df = final_df.drop_duplicates(subset=[
    'mun_istat', 'zone', 'year_semester', 'condition', 'type'
    ], keep='first')

In [ ]:
final_df[final_df['mun_name'] == 'TORRE DE BUSI']

,mun_istat,mun_name,year,year_semester,semester,zone,type,condition,buy_min,buy_max
28006,097080,TORRE DE BUSI,2014,2014_S1,1,B2,Residential housing,Normal,800,1200
28007,097080,TORRE DE BUSI,2014,2014_S1,1,B2,Lowcost housing,Normal,550,780
28008,097080,TORRE DE BUSI,2014,2014_S1,1,B2,Garage,Normal,700,950
28009,097080,TORRE DE BUSI,2014,2014_S1,1,B2,Independent houses and villas,Normal,900,1350
28010,097080,TORRE DE BUSI,2014,2014_S1,1,B2,Warehouses,Normal,320,460
...,...,...,...,...,...,...,...,...,...,...
3723608,097080,TORRE DE BUSI,2025,2025_S2,2,B2,Laboratories,Normal,400,610
3723609,097080,TORRE DE BUSI,2025,2025_S2,2,R1,Residential housing,Normal,950,1200
3723610,097080,TORRE DE BUSI,2025,2025_S2,2,R1,Lowcost housing,Normal,600,900
3723611,097080,TORRE DE BUSI,2025,2025_S2,2,R1,Garage,Normal,650,900


Update ISTAT codes

In [ ]:
# ISTAT codes updated to 2025
df_istat = pd.read_csv('datasets/mun_istat_codes.csv')

# ISTAT codes changes
df_change = pd.read_csv('datasets/changes_istat.csv')

In [ ]:
# Uniform ISTAT codes across datasets
add_zeroes(df_istat, ['mun_istat'], 6)
add_zeroes(df_change, ['mun_istat_old', 'mun_istat_new'], 6)

df_istat['mun_istat'] = df_istat['mun_istat'].astype('object')
df_change['mun_istat_old'] = df_change['mun_istat_old'].astype('object')
df_change['mun_istat_new'] = df_change['mun_istat_new'].astype('object')

In [ ]:
final_df.loc[(final_df['mun_name'] == 'TORRE DE BUSI'), ['mun_istat']] = ['016215']

# Update ISTAT codes
updated_df = update_istat(
    df=final_df,
    df_map=df_change, 
    valid_codes=df_istat["mun_istat"], 
    istat_col="mun_istat",
    istat_old = "mun_istat_old",
    istat_new = "mun_istat_new"
)

# Select only suppressed municipalities
supp_df = updated_df[updated_df['suppressed'].isin([True])]

supp_df = supp_df.drop(columns = ['mun_istat', 'mun_istat_updated', 'changed', 'suppressed'])

# Select only non-suppressed municipalities
updated_df = updated_df[updated_df['suppressed'].isin([False])]

updated_df = updated_df.drop(columns = ['mun_istat', 'changed', 'suppressed'])

updated_df = updated_df.rename(columns = {'mun_istat_updated' : 'mun_istat'})

df_istat_new = df_istat[['mun_istat','mun_name']]

In [ ]:
# Merge suppressed municipalities with new ISTAT codes (name based)
compare_df = pd.merge(supp_df, df_istat_new, on = 'mun_name', how = 'left')

# Drop municipalities with no match in new ISTAT codes
compare_df = compare_df.dropna(subset = ['mun_istat'])

compare_df['mun_istat'] = compare_df['mun_istat'].astype('str')

# Merge with updated dataset
all_df = pd.concat([updated_df, compare_df], ignore_index=True)

Import region and province names

In [ ]:
all_df = pd.merge(all_df, df_istat.drop(columns = ['mun_name', 'prov_istat']), how = 'left', on = 'mun_istat')

Missing/0 values

In [ ]:
# Check for missing values
missing_values = all_df.isnull().sum()
print("Missing values in each column:\n", missing_values)

Missing values in each column:
 mun_name            0
year                0
year_semester       0
semester            0
zone                0
type                0
condition         211
buy_min           259
buy_max           259
mun_istat           0
prov_name        3504
reg_name         3504
dtype: int64


In [ ]:
# Deleting missing values
all_df = all_df.dropna()

In [ ]:
# Check for 0s in buy columns
zero_values = (all_df == 0).sum()
print("Zero values in each column:\n", zero_values)

Zero values in each column:
 mun_name             0
year                 0
year_semester        0
semester             0
zone                 0
type                 0
condition            0
buy_min          19303
buy_max          19303
mun_istat            0
prov_name            0
reg_name             0
dtype: Int64


In [ ]:
# Dropping rows with 0 'buy_min/max' values
all_df = all_df[(all_df['buy_min'] != 0) & (all_df['buy_max'] != 0)]

Correct region and province names

In [ ]:
# Save province names for mapping
#pd.Series(sorted(all_df['prov_name'].unique())).to_csv('datasets/TEMP_province_names.csv', index=False)

# Save region names for mapping
#pd.Series(sorted(all_df['reg_name'].unique())).to_csv('datasets/TEMP_region_names.csv', index=False)

In [ ]:
all_df['prov_name'] = all_df['prov_name'].replace({
    'BARLETTAANDRIATRANI' : 'BARLETTA - ANDRIA - TRANI',
    'BOLZANOBOZEN' : 'BOLZANO - BOZEN',
    'FORLICESENA' : 'FORLI - CESENA',
    'MASSACARRARA' : 'MASSA - CARRARA',
    "VALLE D'AOSTAVALLEE D'AOSTE" : "VALLE D'AOSTA",
    'VERBANOCUSIOOSSOLA' : 'VERBANO - CUSIO - OSSOLA'
})

all_df['reg_name'] = all_df['reg_name'].replace({
    'EMILIAROMAGNA' : 'EMILIA ROMAGNA',
    'FRIULIVENEZIA GIULIA' : 'FRIULI VENEZIA GIULIA',
    'TRENTINOALTO ADIGESUDTIROL' : 'TRENTINO ALTO ADIGE - SUDTIROL',
    "VALLE D'AOSTAVALLEE D'AOSTE" : "VALLE D'AOSTA"
})

Correct the problem with Milan old zones

In [ ]:
# Delete every zone with 0 as second character for Milan
all_df = all_df[~((all_df['mun_name'] == 'MILANO') & (all_df['zone'].str[1] == '0'))]

Correct garage -> garages in 'type'

In [ ]:
all_df['type'] = all_df['type'].replace({
    'Garage' : 'Garages'
})

Save correspondences of normalised names

In [ ]:
names_df = all_df[['mun_istat', 'mun_name', 'prov_name', 'reg_name']]

names_df = names_df.groupby(['mun_istat']).aggregate({
    'mun_name': 'first',
    'prov_name' : 'first',
    'reg_name' : 'first'
}).reset_index()

names_df.to_csv('datasets/names_corr.csv', index = False)

Sobstitute names with non-normalised ones

In [ ]:
# Load names correspondences
corr_df = pd.read_csv('datasets/names_corr_non_normalised.csv')
corr_df = corr_df.drop(columns = 'prov_istat')
add_zeroes(corr_df, 'mun_istat', 6)

# Drop normalised names
all_df = all_df.drop(columns = ['mun_name', 'reg_name', 'prov_name'])

# Merge with non-normalised names
df_final = pd.merge(all_df, corr_df, on = 'mun_istat', how = 'left')

Save data

In [ ]:
df_final.to_csv("datasets/omi_estimate/omi_estimate.csv", index = False)